# Pipeline de Machine Learning — GlicNutri (referência GlucoBench)

**Projeto:** GlicNutri  
**Equipe:** [Fernando Santos, Caio Bueno, Mateus Costa, João Ricardo, Gustavo Ribeiro]  
**Instituição:** UniCesumar — **Data:** 2026

---

## Dataset de referência: GlucoBench

Este notebook foi **adaptado** para usar o dataset público **[GlucoBench: Glucose Monitoring and Lifestyle Data](https://www.kaggle.com/datasets/omenkj/glucobench-glucose-monitoring-and-lifestyle-data)** (`omenkj/glucobench-glucose-monitoring-and-lifestyle-data`) como referência **mais próxima** do domínio do aplicativo GlicNutri.

**Por que GlucoBench é mais alinhado ao GlicNutri do que o Pima Indians Diabetes?**

- Inclui **monitoramento glicêmico** e variáveis ligadas a **estilo de vida** (refeições, atividade, insulinoterapia em contexto de uso real), em geral com **estrutura temporal** (datas/horários).
- O dataset **Pima Indians Diabetes** é uma tabela clínica estática com variáveis como gestações, pressão e espessura de pele — **não** reproduz o uso contínuo do app nem registros temporais típicos de CGM e hábitos.

**O que foi removido como referência principal**

- Download via `jamaltariqcheema/pima-indians-diabetes-dataset` e arquivo `diabetes.csv`.
- Modelagem PyCaret baseada nas colunas **Pregnancies, Glucose (uma coluna por linha estática), BloodPressure, SkinThickness, Outcome**, etc.

---

### Pendências (antes do pipeline completo)

| Item | Status |
|------|--------|
| Seleção do arquivo CSV (ou tabular) correto dentro do GlucoBench | Abrir `FILE_PATH` após listar arquivos |
| Confirmação das colunas reais | Ver células de inspeção e validação |
| Engenharia de features e alinhamento ao Supabase | Após inspeção |
| Definição da variável-alvo (classificação/regressão) | Pendente |
| Treino com PyCaret / pipelines | **Não executar** até validar esquema |

---


---
## 1. Instalação e imports


In [10]:
# ============================================================
# Dependências: kagglehub + stack de análise
# (PyCaret ficará para etapa posterior, após confirmar colunas.)
# ============================================================

!pip install kagglehub[pandas-datasets] --quiet
!pip install pandas numpy matplotlib seaborn openpyxl --quiet

print("Instalação concluída. Reinicie o runtime do Colab/Jupyter se o kernel solicitar.")



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Instalação concluída. Reinicie o runtime do Colab/Jupyter se o kernel solicitar.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# ============================================================
# Imports gerais (sem PyCaret até definirmos features e alvo)
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

print("Imports OK — pandas", pd.__version__)


Imports OK — pandas 3.0.1


---
## 2. Download e listagem do dataset GlucoBench (Kaggle)

A célula abaixo **baixa** o dataset via `kagglehub`, define **`dataset_root`** (caminho absoluto da pasta no cache) e **lista todos os arquivos** com **caminho completo** e, em seguida, o caminho **relativo** a `dataset_root`.  
Não é necessário configurar `FILE_PATH` aqui; o `df` será criado depois, quando você escolher qual arquivo abrir.


In [12]:
# ============================================================
# Download + listagem + carregamento opcional (pandas)
# Slug Kaggle: omenkj/glucobench-glucose-monitoring-and-lifestyle-data
# ============================================================

import os
import kagglehub
from kagglehub import KaggleDatasetAdapter

DATASET_SLUG = "omenkj/glucobench-glucose-monitoring-and-lifestyle-data"

# >>> EDITAR AQUI após ver a listagem de arquivos na saída desta célula:
# Caminho relativo ao diretório raiz do dataset no cache Kaggle (use / ou \\ conforme mostrado).
FILE_PATH = "./GlucoBench_benchmark_dataset.csv"

print("Baixando dataset (cache local Kagglehub)...")
dataset_root = kagglehub.dataset_download(DATASET_SLUG)
print("\\nPasta local do dataset:", dataset_root)

print("\\n--- Arquivos no dataset (recursivo) ---")
all_files = []
for walk_root, _dirs, files in os.walk(dataset_root):
    for fname in files:
        full = os.path.join(walk_root, fname)
        rel = os.path.relpath(full, dataset_root)
        all_files.append(rel)
        print(rel)

df = None
if not str(FILE_PATH).strip():
    print(
        "\\n⚠️ FILE_PATH está vazio. Escolha o CSV/arquivo tabular adequado na listagem acima, "
        "preencha FILE_PATH com o caminho relativo e reexecute esta célula para criar `df`."
    )
else:
    print("\\nCarregando via kagglehub.load_dataset →", FILE_PATH)
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        DATASET_SLUG,
        FILE_PATH,
    )
    print("Shape:", getattr(df, "shape", None))
    display(df.head())



Baixando dataset (cache local Kagglehub)...
\nPasta local do dataset: C:\Users\flima\.cache\kagglehub\datasets\omenkj\glucobench-glucose-monitoring-and-lifestyle-data\versions\1
\n--- Arquivos no dataset (recursivo) ---
GlucoBench_benchmark_dataset.csv
\nCarregando via kagglehub.load_dataset → ./GlucoBench_benchmark_dataset.csv
Shape: (15731, 31)


,user_id,timestamp,glucose,cgm_quality_flag,insulin_bolus,insulin_basal,carbs,meal_type,exercise_steps,exercise_intensity,heart_rate,skin_temp,gsr,sleep_stage,medication_other,stress_level,alcohol,notes,device_id,hbA1c,age,sex,weight,carb_ratio,insulin_sensitivity,timezone,region,glucose_lag_1,glucose_lag_3,glucose_lag_6,glucose_roll_mean_1h
0,U001,2024-09-01 00:00:00,127.0000,1,0.0000,1.0500,0,none,0,none,102.8000,35.7000,0.6100,awake,steroid,3,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,NaN,NaN,NaN,127.0000
1,U001,2024-09-01 00:15:00,126.5000,1,0.0000,0.8400,0,none,0,none,91.0000,37.5000,1.2200,light,antihistamine,1,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,127.0000,NaN,NaN,126.7500
2,U001,2024-09-01 00:20:00,126.8000,1,0.0000,1.1900,0,none,0,none,69.9000,35.9000,2.2900,REM,none,0,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,126.5000,NaN,NaN,126.7667
3,U001,2024-09-01 00:25:00,127.2000,1,0.0000,0.9900,0,none,0,none,106.3000,35.9000,2.1800,deep,none,5,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,126.8000,127.0000,NaN,126.8750
4,U001,2024-09-01 00:30:00,126.7000,1,0.0000,1.1200,0,none,0,none,102.0000,37.4000,1.9500,light,none,2,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,127.2000,126.5000,NaN,126.8400


In [13]:
df.columns

Index(['user_id', 'timestamp', 'glucose', 'cgm_quality_flag', 'insulin_bolus',
       'insulin_basal', 'carbs', 'meal_type', 'exercise_steps',
       'exercise_intensity', 'heart_rate', 'skin_temp', 'gsr', 'sleep_stage',
       'medication_other', 'stress_level', 'alcohol', 'notes', 'device_id',
       'hbA1c', 'age', 'sex', 'weight', 'carb_ratio', 'insulin_sensitivity',
       'timezone', 'region', 'glucose_lag_1', 'glucose_lag_3', 'glucose_lag_6',
       'glucose_roll_mean_1h'],
      dtype='str')

In [14]:
df.describe()

,glucose,cgm_quality_flag,insulin_bolus,insulin_basal,carbs,exercise_steps,heart_rate,skin_temp,gsr,stress_level,alcohol,notes,hbA1c,age,weight,carb_ratio,insulin_sensitivity,glucose_lag_1,glucose_lag_3,glucose_lag_6,glucose_roll_mean_1h
count,15731.0000,15731.0000,15731.0000,15731.0000,15731.0000,15731.0000,15731.0000,15731.0000,15731.0000,15731.0000,15731.0000,0.0000,15731.0000,15731.0000,15731.0000,15731.0000,15731.0000,15721.0000,15701.0000,15671.0000,15731.0000
mean,112.6906,0.9893,0.2481,0.9999,2.9559,75.4390,85.2306,36.5034,1.4969,2.5108,0.0213,NaN,7.2715,42.6452,79.6652,10.9981,47.8193,112.6815,112.6633,112.6352,112.6368
std,30.2982,0.1031,1.2449,0.1147,13.6956,492.8020,14.5255,0.5767,0.5802,1.7015,0.1688,NaN,0.6958,18.9335,18.2138,1.8989,4.7548,30.2927,30.2817,30.2645,30.1968
min,70.0000,0.0000,0.0000,0.8000,0.0000,0.0000,60.0000,35.5000,0.5000,0.0000,0.0000,NaN,6.1000,19.0000,51.0000,9.0000,40.0000,70.0000,70.0000,70.0000,70.0000
25%,87.5000,1.0000,0.0000,0.9000,0.0000,0.0000,72.5000,36.0000,0.9900,1.0000,0.0000,NaN,6.6000,24.0000,60.0000,9.0000,44.0000,87.5000,87.5000,87.5000,87.5333
50%,108.0000,1.0000,0.0000,1.0000,0.0000,0.0000,85.3000,36.5000,1.4900,3.0000,0.0000,NaN,7.5000,35.0000,79.0000,10.0000,48.0000,108.0000,108.0000,108.0000,107.9000
75%,131.6000,1.0000,0.0000,1.1000,0.0000,0.0000,97.9000,37.0000,2.0000,4.0000,0.0000,NaN,7.9000,61.0000,99.0000,12.0000,52.0000,131.6000,131.6000,131.6000,131.3167
max,198.4000,1.0000,11.0000,1.2000,90.0000,4952.0000,110.0000,37.5000,2.5000,5.0000,2.0000,NaN,8.3000,70.0000,99.0000,15.0000,55.0000,198.4000,198.4000,198.4000,197.5000


In [15]:
df.isnull().sum()

user_id                     0
timestamp                   0
glucose                     0
cgm_quality_flag            0
insulin_bolus               0
insulin_basal               0
carbs                       0
meal_type                   0
exercise_steps              0
exercise_intensity          0
heart_rate                  0
skin_temp                   0
gsr                         0
sleep_stage                 0
medication_other            0
stress_level                0
alcohol                     0
notes                   15731
device_id                   0
hbA1c                       0
age                         0
sex                         0
weight                      0
carb_ratio                  0
insulin_sensitivity         0
timezone                    0
region                      0
glucose_lag_1              10
glucose_lag_3              30
glucose_lag_6              60
glucose_roll_mean_1h        0
dtype: int64

In [16]:
# ============================================================
# Inspeção inicial (somente se df foi carregado)
# ============================================================

if df is None:
    print("df ainda não foi carregado — use a listagem da célula anterior e abra o CSV em uma célula dedicada quando quiser.")
else:
    display(df.head())
    print("\n--- info() ---")
    df.info()
    print("\n--- Colunas ---")
    print(list(df.columns))
    print("\n--- Valores nulos por coluna ---")
    display(df.isnull().sum())


,user_id,timestamp,glucose,cgm_quality_flag,insulin_bolus,insulin_basal,carbs,meal_type,exercise_steps,exercise_intensity,heart_rate,skin_temp,gsr,sleep_stage,medication_other,stress_level,alcohol,notes,device_id,hbA1c,age,sex,weight,carb_ratio,insulin_sensitivity,timezone,region,glucose_lag_1,glucose_lag_3,glucose_lag_6,glucose_roll_mean_1h
0,U001,2024-09-01 00:00:00,127.0000,1,0.0000,1.0500,0,none,0,none,102.8000,35.7000,0.6100,awake,steroid,3,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,NaN,NaN,NaN,127.0000
1,U001,2024-09-01 00:15:00,126.5000,1,0.0000,0.8400,0,none,0,none,91.0000,37.5000,1.2200,light,antihistamine,1,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,127.0000,NaN,NaN,126.7500
2,U001,2024-09-01 00:20:00,126.8000,1,0.0000,1.1900,0,none,0,none,69.9000,35.9000,2.2900,REM,none,0,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,126.5000,NaN,NaN,126.7667
3,U001,2024-09-01 00:25:00,127.2000,1,0.0000,0.9900,0,none,0,none,106.3000,35.9000,2.1800,deep,none,5,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,126.8000,127.0000,NaN,126.8750
4,U001,2024-09-01 00:30:00,126.7000,1,0.0000,1.1200,0,none,0,none,102.0000,37.4000,1.9500,light,none,2,0.0000,NaN,Medtronic_670G,6.1000,49,M,79,12,42,America/New_York,USA,127.2000,126.5000,NaN,126.8400



--- info() ---
<class 'pandas.DataFrame'>
RangeIndex: 15731 entries, 0 to 15730
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   user_id               15731 non-null  str    
 1   timestamp             15731 non-null  str    
 2   glucose               15731 non-null  float64
 3   cgm_quality_flag      15731 non-null  int64  
 4   insulin_bolus         15731 non-null  float64
 5   insulin_basal         15731 non-null  float64
 6   carbs                 15731 non-null  int64  
 7   meal_type             15731 non-null  str    
 8   exercise_steps        15731 non-null  int64  
 9   exercise_intensity    15731 non-null  str    
 10  heart_rate            15731 non-null  float64
 11  skin_temp             15731 non-null  float64
 12  gsr                   15731 non-null  float64
 13  sleep_stage           15731 non-null  str    
 14  medication_other      15731 non-null  str    
 15  stress_level  

user_id                     0
timestamp                   0
glucose                     0
cgm_quality_flag            0
insulin_bolus               0
insulin_basal               0
carbs                       0
meal_type                   0
exercise_steps              0
exercise_intensity          0
heart_rate                  0
skin_temp                   0
gsr                         0
sleep_stage                 0
medication_other            0
stress_level                0
alcohol                     0
notes                   15731
device_id                   0
hbA1c                       0
age                         0
sex                         0
weight                      0
carb_ratio                  0
insulin_sensitivity         0
timezone                    0
region                      0
glucose_lag_1              10
glucose_lag_3              30
glucose_lag_6              60
glucose_roll_mean_1h        0
dtype: int64

In [17]:
# ============================================================
# Validação: presença de temas (por nome de coluna — sem inventar dados)
# ============================================================

if df is None:
    print("Sem DataFrame — nada a validar.")
else:
    cols_lower = {str(c).lower(): c for c in df.columns}

    def hits(keywords):
        found = []
        for kw in keywords:
            for low, orig in cols_lower.items():
                if kw in low and orig not in found:
                    found.append(orig)
        return found

    groups = {
        "glucose / glicemia / CGM": hits(
            ["glucose", "gluc", "bg", "cgm", "interstitial", "mg/dl", "mgdl", "sensor"]
        ),
        "tempo (data/hora/timestamp)": hits(
            ["time", "date", "timestamp", "datetime", "hour", "day"]
        ),
        "insulina": hits(["insulin", "insulina", "bolus", "basal"]),
        "refeição / carboidratos / alimento": hits(
            ["meal", "food", "carb", "nutrition", "kcal", "calorie"]
        ),
        "atividade / lifestyle": hits(
            ["activity", "step", "exercise", "sleep", "heart", "lifestyle"]
        ),
    }

    print("Colunas detectadas por tema (heurística sobre o *nome* da coluna):\n")
    for tema, cols in groups.items():
        print(f"• {tema}: {cols if cols else '— nenhuma correspondência pelo nome —'}")


Colunas detectadas por tema (heurística sobre o *nome* da coluna):

• glucose / glicemia / CGM: ['glucose', 'glucose_lag_1', 'glucose_lag_3', 'glucose_lag_6', 'glucose_roll_mean_1h', 'cgm_quality_flag']
• tempo (data/hora/timestamp): ['timestamp', 'timezone']
• insulina: ['insulin_bolus', 'insulin_basal', 'insulin_sensitivity']
• refeição / carboidratos / alimento: ['meal_type', 'carbs', 'carb_ratio']
• atividade / lifestyle: ['exercise_steps', 'exercise_intensity', 'sleep_stage', 'heart_rate']


---
## 3. Mapeamento conceitual GlucoBench → GlicNutri (Supabase)

Esta tabela é **conceitual**: os nomes reais das colunas dependem do arquivo escolhido no GlucoBench. Use-a para planejar ETL e joins com o banco do app.

| Conceito GlicNutri | Tabela / origem no app | Como relacionar ao GlucoBench |
|--------------------|-------------------------|--------------------------------|
| Identidade do usuário | `paciente` | IDs ou chaves de participante no GlucoBench (se existirem) → `id_paciente_uuid` após anonimização/regra de projeto |
| Glicemia manual | `registro_glicemia_manual` | Colunas de valor + data/hora de glicose **self-monitoring** |
| CGM / sensor | `api_libreview_cgm` *(se existir no seu projeto)* | Séries temporais / valores intersticiais — alinhar timestamps |
| Medicação / insulina | `registro_medicacao` | Eventos de dose / tipo — mapear colunas de insulina/bolus no GlucoBench |
| Refeição / macros | `refeicao_ia` | Refeições, carboidratos, energia — similar a features de meal/nutrition no GlucoBench |
| Rotina / hábitos | `diario_rotina` *(nem sempre existe como tabela)* | Atividade, sono, passos — se o notebook de dados do app usar estado serializado, documentar no ETL |

**Nota:** No repositório atual, `api_libreview_cgm` e `diario_rotina` podem não estar versionados como tabelas nas migrations — validar no Supabase antes do merge de dados.

---


---
## Anexo — código legado Pima Indians Diabetes

O pipeline antigo baseado em **Pima Indians Diabetes** (EDA, PyCaret, clustering, etc.) foi **isolado** para não competir com o fluxo principal GlucoBench neste ficheiro.

Consultar: **[Diabetes_pipeline_ml_legacy_pima_anexo.ipynb](./Diabetes_pipeline_ml_legacy_pima_anexo.ipynb)**.
